# Lekcia 17 - Vytváranie lokálnych AI agentov pomocou Foundry Local a Qwen

V tomto notebooku vytvoríte **lokálneho inžinierskeho asistenta**, ktorý beží úplne na vašom pracovnom počítači. Nebola použitá žiadna cloudová inferencia. Asistent môže:

1. **Volanie nástrojov** cez volanie funkcií Qwen prostredníctvom Foundry Local.
2. **Zoznamovať a čítať súbory** v rámci sandboxovaného projektového adresára.
3. **Analyzovať kód** pomocou jednoduchých metrík.
4. **Vyhľadávať v dokumentácii** s použitím lokálneho RAG (Chroma).
5. **Používať lokálny MCP server** (ak nie je žiaden nakonfigurovaný, funkcia sa elegantne preskočí).

Kód agenta vyzerá takmer rovnako ako v cloudových lekciách — iba klientsky endpoint sa presúva z cloudu na `localhost`.


## Nastavenie

Pred spustením tohto zápisníka:

1. **Nainštalujte Microsoft Foundry Local** (pozrite si [dokumentáciu](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) pre váš operačný systém).
2. **Stiahnite a spustite model Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Nainštalujte nižšie uvedené Python balíčky.

Všetko beží lokálne. Počítač s ~8 GB RAM je realistickým minimom.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Pripojenie k Foundry Local

`FoundryLocalManager` stiahne model, ak je to potrebné, spustí lokálnu službu a poskytne nám **endpoint kompatibilný s OpenAI**. Potom naň nasmerujeme štandardné OpenAI SDK. Kľúč API je lokálny zástupný symbol — nedochádza k používaniu žiadnych cloudových poverení.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Lokálne nástroje (sandboxované operácie so súbormi)

Vytvoríme malý ukážkový projekt na disku a potom definujeme nástroje obmedzené na koreň tohto projektu. Sandbox kontrola je dôležitá aj lokálne: nástroj, ktorý číta ľubovoľné cesty, beží s oprávneniami vášho používateľa a môže pristupovať ku všetkému, čo môžete vy.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. Lokálny RAG s Chromou

Zahrnieme malú množinu úryvkov dokumentácie do lokálnej kolekcie Chroma. Chroma beží v rámci procesu a ukladá vektory na disk — žiadny server, žiadny cloud. Nástroj `search_docs` načíta najrelevantnejšie úryvky pre dopyt.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. Slučka volania nástrojov

Teraz zaregistrujeme nástroje v modeli pomocou schémy OpenAI nástrojov a spustíme štandardnú slučku volania nástrojov — model si vyžiada nástroj, my ho lokálne vykonáme, výsledok naspäť poskytneme a opakujeme, kým model nevytvorí konečnú odpoveď. Spoľahlivé volanie funkcií Qwen-a je to, čo umožňuje, že to funguje priamo na zariadení.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. Lokálny MCP (voliteľné)

MCP je transport, nie cloudová služba — server MCP môže bežať ako lokálny proces cez `stdio`. Nižšie uvedená bunka ukazuje, ako by ste sa pripojili k lokálnemu MCP serveru, ak ho máte nakonfigurovaný (napríklad súborový systém server). Pri neprítomnosti nastavenia `LOCAL_MCP_COMMAND` sa bezpečne preskočí, takže zápisník sa stále vykoná od začiatku do konca bez neho.

Bezpečnostná poznámka: lokálny MCP server beží s oprávneniami vášho používateľa. Obmedzte jeho rozsah na adresár projektu a pred použitím overte jeho výstupy.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Zhrnutie

Vybudovali ste inžinierskeho asistenta, ktorý beží úplne na vašom zariadení:

- **Foundry Local** poskytoval model **Qwen** za OpenAI-kompatibilným endpointom — takže agentový kód zodpovedá cloudovým lekciám.
- **Sandboxed tools** umožnili agentovi prístup k súborom a analýzu kódu bez opustenia adresára projektu.
- **Chroma** poskytovala **lokálny RAG** nad dokumentáciou.
- **Local MCP** ukázal, ako znovu využiť MCP ekosystém offline.

Nebola použitá žiadna inferencia v cloude.

### Výzva

Rozšírte lokálneho agenta, aby:

1. **pracoval s viacerými MCP servermi** — pripojte filesystem server a git server a nechajte agenta vybrať si medzi nimi.
2. **používal lokálnu pamäť** — ukladajte krátku históriu konverzácie na disk, aby si asistent pamätal predchádzajúce kroky aj po reštartech poznámkového bloku.
3. **podporoval lokálnu multi-agent orchestráciu** — pridajte druhého lokálneho agenta (napr. recenzenta) a nechajte ich spolupracovať na úlohe.

V ďalšej lekcii sa naučíte, ako zabezpečiť nasadených AI agentov.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
